# Mise en page scientifique et atlas

**Cartomize · Version 1.0**

Durée indicative : 35 minutes.

## Objectifs

Configurer les cadres, une légende, un tableau et un graphique ; exporter plusieurs formats et produire une page par secteur.

## Apport de Cartomize

Une configuration cartographique peut être réutilisée dans 24 maquettes et propagée à un atlas sans refaire chaque page.

Données : paysage pédagogique généré avec une graine fixe. Les cartes et mesures ne décrivent aucun site réel.

## Préparation

Installer l’environnement décrit dans le [guide de démarrage](../README.md), puis exécuter les cellules dans l’ordre. Chaque exécution utilise un nouveau dossier de résultats.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display, Image
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.py').is_file() or (p/'tutorials/course.py').is_file())
root = root if (root/'course.py').is_file() else root/'tutorials'
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import cartomize as cm
from course import demo, run_directory, show_raster, show_rgb, gallery, CRS, CLASSES, ROLES

In [2]:
out = run_directory("07_mise_en_page_atlas")
d = demo(out/"data")

## 1. Choisir une maquette selon le contenu

Le catalogue indique les formats et cadres. Un cadre de localisation doit recevoir une emprise pertinente ; répéter la même vue dans tous les cadres apporte peu d’information.

In [3]:
templates=pd.DataFrame(cm.list_templates())
display(templates[['id','name','map_frames']])
identifier='transport/accessibilite-reseau-a4'
plan=cm.layout_plan(identifier)
display(pd.DataFrame([{'id':i.item_id,'type':i.kind} for i in plan.items]))

,id,name,map_frames
0,administrative/institutionnel,Carte administrative: Institutionnel,2
1,professionnelles/19-carte-verticale-encart-sit...,Carte verticale avec encart de situation,2
2,professionnelles/09-localisation-deux-encarts,Localisation avec deux encarts,3
3,professionnelles/03-localisation-hierarchique,Localisation hiérarchique,3
4,professionnelles/25-localisation-territoriale-...,Localisation territoriale en trois niveaux,3
5,agriculture/aptitude-agricole-a3,Aptitude agricole et systèmes de production,2
6,biodiversite/connectivite-ecologique-a4,Biodiversité et connectivité écologique,2
7,demographie/population-densite-a4,Population et densité: Choroplèthe,2
8,energie/electrification-a4,Accès à l’électricité et infrastructures,2
9,professionnelles/15-carte-deux-fenetres-detail,Carte avec deux fenêtres de détail,3


,id,type
0,title-rule,shape
1,locator,map_frame
2,main-map,map_frame
3,legend,legend
4,network-summary,table
5,north-600-410,north_arrow
6,scale-382-446,scale_bar
7,subtitle,text
8,title,title
9,travel-profile,chart


## 2. Remplir les éléments de la page

In [4]:
carte=cm.Map(template=identifier,title='Structure du paysage',subtitle='Exercice cartographique',crs=CRS,credits='Cartomize | Données synthétiques | UTM 33S')
carte.add_layer(d['before'],name='Couverture simulée',classes=CLASSES)
carte.add_layer(d['points'],name='Stations',labels='name',color='#202020')
carte.add_layer(d['zones'],name='Secteurs',color='none',edgecolor='#333333',legend=False)
zones=cm.read_file(d['zones'])
for frame in carte.frame_ids:
    if frame!=plan.primary_map_id:
        carte.set_frame(frame,extent=tuple(zones.iloc[[0]].total_bounds))
areas=cm.raster.class_areas(d['before'],unit='ha')
areas['Classe']=areas['class'].map({1:'Végétation',2:'Sol',3:'Eau'})
areas=areas[['Classe','area_ha']].rename(columns={'area_ha':'Surface (ha)'})
areas['Surface (ha)']=areas['Surface (ha)'].round(1)
for item in plan.items:
    if item.kind=='table':carte.set_table(item.item_id,areas)
    elif item.kind=='chart':carte.set_chart(item.item_id,areas.iloc[:,0].astype(str).tolist(),areas.iloc[:,-1].tolist())
    elif item.kind=='text' and item.item_id not in ('title','subtitle','credits'):
        carte.set_text(item.item_id,'Données simulées' if item.height_mm>8 else 'Exercice')
carte.set_text('subtitle','Exercice cartographique | Paysage simulé')
carte.set_item('travel-profile',height_mm=22,content={'title':'Superficie par classe (ha)'})
carte.set_item('network-summary',content={'title':'Superficies'})
for extension in ('pdf','png','svg'):
    carte.export(out/f'planche.{extension}',dpi=220)
fig=carte.render(dpi=120);gallery(fig,'07_planche');plt.show()

## 3. Produire et contrôler l’atlas

La nomenclature est partagée par toutes les pages. Le nom des secteurs identifie les fichiers produits.

In [5]:
atlas=cm.atlas(carte,zones,out/'atlas',name_column='name',dpi=150)
assert len(list((out/'atlas').glob('*.pdf')))==len(zones)
display(carte.audit())

{'schema': 'cartomize.map.audit.v1',
 'valid': True,
 'issues': [],
 'layer_plan': [{'name': 'Couverture simulée',
   'kind': 'raster',
   'role': 'landcover',
   'labels': None,
   'zorder': 10,
   'insertion_index': 0},
  {'name': 'Secteurs',
   'kind': 'vector',
   'role': 'polygon',
   'labels': None,
   'zorder': 20,
   'insertion_index': 2},
  {'name': 'Stations',
   'kind': 'vector',
   'role': 'points',
   'labels': 'name',
   'zorder': 65,
   'insertion_index': 1}],
 'scope': 'Geometry validity, fields, full raster class coverage and layout metadata.',
 'limitations': ['Not a certification of scientific accuracy or completeness.',
  'Raster emptiness uses a sample; class coverage is checked by blocks.',
  'Label collisions, text overflow and all spatial relations are not exhaustively checked.']}

## 4. Adapter les autres maquettes

Le même principe s’applique aux 24 maquettes : lire les identifiants dans `layout_plan`, définir les cadres et renseigner les éléments utiles. Le titre thématique d’une maquette ne réalise pas l’analyse correspondante. La maquette « fragmentation » ne calcule pas d’indice de fragmentation.

## Exercice

Créer une carte sans maquette puis un atlas à partir de cette carte. Quels paramètres doivent rester identiques pour comparer les pages ?

<details>
<summary>Éléments de correction</summary>

Conserver les libellés, couleurs, seuils et unités. Préciser si l’échelle est commune ou adaptée à chaque secteur. Vérifier la légende, l’échelle, les sources et le SCR avant diffusion.

</details>

## Reproduction

Les paramètres, la graine et les chemins sont explicites dans les cellules. Le répertoire `out` contient les produits de cette exécution. Adapter les sources, le système de coordonnées et les paramètres avant de transférer la méthode à une étude.